# Apache Iceberg - JavaScript

All 19 JavaScript examples from [docs/iceberg.md](https://platob.github.io/yggdryl/iceberg/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('venue: utf8')], {
  nullable: false,
})

const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

// A table is created in a folder, and a folder is all it ever touches.
const table = iceberg.Table.create(root, schema, ['venue'])

// A table that has never been written to has no current snapshot.
assert.equal(table.currentSnapshot, null)
assert.equal(table.scan().toTable().numRows, 0)

table.append(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    venue: arrow.vectorFromArray(['XNAS', 'XNYS'], new arrow.Utf8()),
  }),
)

assert.equal(table.currentSnapshot.operation, 'append')
assert.equal(table.dataFiles().length, 2, 'one file per venue')

// Reopening finds the table again, with no catalog in between.
const reopened = iceberg.Table.open(root)
assert.equal(reopened.scan().toTable().numRows, 2)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## What a table writes

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, schema)
table.append(new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) }))

// `table.root` is the folder handle the table reads and writes through.
const names = [...table.root.ls(true)]
  .filter((entry) => entry.isFile())
  .map((entry) => entry.name)

// One Parquet data file, one manifest, one manifest list, two metadata
// documents (create, then commit), and the version hint that finds them.
assert.ok(names.some((name) => name.endsWith('.parquet')))
assert.ok(names.some((name) => name.startsWith('snap-') && name.endsWith('.avro')))
assert.ok(names.some((name) => name.endsWith('-m0.avro')))
assert.ok(names.includes('v1.metadata.json'))
assert.ok(names.includes('v2.metadata.json'))
assert.ok(names.includes('version-hint.text'))

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Manifest lists and manifests

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('venue: utf8')], {
  nullable: false,
})
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, schema, ['venue'])
table.append(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    venue: arrow.vectorFromArray(['XNAS', 'XNAS'], new arrow.Utf8()),
  }),
)

// A snapshot names one manifest list; each of its rows is a manifest.
const manifests = table.manifests()
assert.equal(manifests.length, 1)
assert.equal(manifests[0].content, 'data')
assert.equal(manifests[0].addedFilesCount, 1)
assert.equal(manifests[0].addedRowsCount, 2)

// Each manifest row is a data file plus what the writer measured about it.
const [file] = table.dataFiles()
assert.equal(file.fileFormat, 'PARQUET')
assert.equal(file.recordCount, 2)
assert.deepEqual(file.partitionNames, ['venue'])

// Statistics are keyed by field id, which is what lets a planner skip a file.
assert.ok(file.valueCounts.some((entry) => entry.fieldId === 1 && entry.count === 2))
assert.ok(file.columnSizes.some((entry) => entry.fieldId === 1))

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Partition specs and the Hive layout

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('venue: utf8')], {
  nullable: false,
})
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, schema, ['venue'])
table.append(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    venue: arrow.vectorFromArray(['XNAS', null], new arrow.Utf8()),
  }),
)

const files = table.dataFiles()
assert.equal(files.length, 2)
const absent = files.find((file) => file.partition[0].asJs() === null)
assert.ok(absent.filePath.includes('venue=null'), 'the path spells it')
assert.equal(absent.partition[0].asJs(), null, 'the manifest means it')

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Reading with column pushdown

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('symbol: utf8')], {
  nullable: false,
})
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, schema)
table.append(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
  }),
)

// The target names the columns to keep; each file's Parquet reader gets it as
// its own projection mask, so the dropped column chunk is never decoded.
const wanted = fields.struct('row', [schema.dataType.at(0)], { nullable: false })
const projected = table.scan(wanted).toTable()
assert.deepEqual(projected.schema.fields.map((child) => child.name), ['id'])
assert.equal(projected.numRows, 2)

// No target reads everything.
assert.equal(table.scan().toTable().numCols, 2)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Time travel and the inspection tables

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')

const table = iceberg.Table.create(root, schema)
table.append(new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) }))
const past = table.currentSnapshot.snapshotId
table.overwrite(new arrow.Table({ id: arrow.vectorFromArray([9n], new arrow.Int64()) }))

// The present shows the overwrite; the retained snapshot shows what was.
assert.deepEqual(table.scan().toTable().getChild('id').toArray(), BigInt64Array.from([9n]))
assert.deepEqual(table.scanAt(past).toTable().getChild('id').toArray(), BigInt64Array.from([1n]))

// A branch or tag resolves by name, and every commit moves `main`.
assert.equal(table.snapshotByRef('main').snapshotId, table.currentSnapshot.snapshotId)

// The inspection readers render the table's own record as record batches.
assert.equal(table.inspectHistory().toTable().numRows, 2)
assert.equal(table.inspectSnapshots().toTable().getChild('operation').get(1), 'overwrite')
assert.equal(table.inspectFiles().toTable().numRows, 1)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Filtered reads and filtered writes

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('venue: utf8'), Field.from('qty: int64')],
  { nullable: false },
)
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')
const table = iceberg.Table.create(root, schema, ['venue'])

const rows = (ids, venues, quantities) =>
  new arrow.Table({
    id: arrow.vectorFromArray(ids, new arrow.Int64()),
    venue: arrow.vectorFromArray(venues, new arrow.Utf8()),
    qty: arrow.vectorFromArray(quantities, new arrow.Int64()),
  })

// One commit per venue, so the manifest list has three rows to prune.
for (const [id, venue] of [[1n, 'XNAS'], [2n, 'XNYS'], [3n, 'XLON']]) {
  table.append(rows([id], [venue], [10n]))
}
const inserted = table.currentSnapshot.snapshotId

// Nothing is listed and no data file is opened: the manifest list's
// per-partition summaries exclude two manifests before either is read.
const plan = table.plan({ venue: 'XNYS' })
assert.equal(plan.filesPlanned, 1)
assert.equal(plan.recordCount, 1)
assert.equal(plan.manifestsRead, 1)
assert.equal(plan.manifestsSkipped, 2)
assert.equal(table.scanWhere({ venue: 'XNYS' }).toTable().numRows, 1)

// A filter on a column the spec does not partition on prunes on the file's
// own recorded bounds instead, then filters the rows the survivors hold.
assert.equal(table.plan([{ column: 'id', value: '3' }]).filesSkipped, 2)

// A filtered overwrite replaces the files the filter selects and carries
// every other file into the new snapshot at its own path, statistics and all.
const paths = () => new Set(table.dataFiles().map((file) => file.filePath))
const before = paths()
table.overwriteWhere({ venue: 'XNYS' }, rows([2n], ['XNYS'], [99n]))
const after = paths()
assert.equal([...before].filter((file) => !after.has(file)).length, 1)
assert.equal([...before].filter((file) => after.has(file)).length, 2)

// A merge upserts on the key: 3 is stored and updates, 4 is new and appends.
table.merge(rows([3n, 4n], ['XLON', 'XLON'], [7n, 8n]), ['id'])
const merged = new Map(table.scan().toTable().toArray().map((row) => [row.id, row.qty]))
assert.deepEqual([...merged.keys()].sort(), [1n, 2n, 3n, 4n])
assert.equal(merged.get(2n), 99n)
assert.equal(merged.get(4n), 8n)

// Narrowed first: a merge into one partition can read no other partition.
table.mergeWhere({ venue: 'XNAS' }, rows([1n], ['XNAS'], [42n]), ['id'])
assert.equal(table.scanWhere({ venue: 'XNAS' }).toTable().getChild('qty').get(0), 42n)

// History plans the same way: the snapshot before the overwrite still
// selects one file for that partition, and it is the file that held 10.
assert.equal(table.planAt(inserted, { venue: 'XNYS' }).filesPlanned, 1)
assert.equal(
  table.scanAt(inserted, { venue: 'XNYS' }).toTable().getChild('qty').get(0),
  10n,
)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## The three record methods over a table

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('venue: utf8?')], {
  nullable: false,
})
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')
iceberg.Table.create(root, schema, ['venue'])

const rows = (ids, venues) =>
  BatchReader.from(
    new arrow.Table({
      id: arrow.vectorFromArray(ids, new arrow.Int64()),
      venue: arrow.vectorFromArray(venues, new arrow.Utf8()),
    }),
  )

// The folder *is* the table, so the ordinary record surface reaches it. Its
// options come from the metadata, before a single data file exists.
const folder = IOBase.from(root)
const options = folder.recordOptions()
folder.writeArrowBatchReader(rows([1n, 2n], ['XNAS', 'XNYS']), options)
folder.appendArrowBatchReader(rows([3n], ['XLON']), options)

// A match key upserts: `2` is stored and updates, `9` is new and appends.
folder.writeArrowBatchReader(
  rows([2n, 9n], ['XNYS', 'XLON']),
  options.withMergeByNames(['id']),
)

assert.equal(folder.readArrowBatchReader(options).toTable().numRows, 4)

// Each call was one commit, and the read went through the last one.
assert.equal(iceberg.Table.open(root).snapshots.length, 3)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## A warehouse of tables

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const warehouse = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const catalog = new iceberg.Catalog(warehouse)

// The explicit spelling: the schema is numbered here, and its partition
// marks become the identity spec.
const marked = fields
  .struct('row', [Field.from('id: int64'), Field.from('venue: utf8')], { nullable: false })
  .withPartitionFields(['venue'])
catalog.createTable('nyc.trades', marked)

const rows = (ids, venues) =>
  new arrow.Table({
    id: arrow.vectorFromArray(ids, new arrow.Int64()),
    venue: arrow.vectorFromArray(venues, new arrow.Utf8()),
  })
const table = catalog.append('nyc.trades', rows([1n, 2n], ['XNAS', 'XNYS']))
assert.equal(table.scan().toTable().numRows, 2)
assert.equal(catalog.append('nyc.trades', rows([3n], ['XNAS'])).scan().toTable().numRows, 3)

// The dotted name is the folder nyc/trades, and the marks became the spec.
assert.ok(catalog.hasTable('nyc.trades'))
assert.deepEqual(catalog.table('nyc.trades').spec.fields.map((field) => field.name), ['venue'])
assert.deepEqual(catalog.listNamespaces(), ['nyc'])
assert.deepEqual(catalog.listTables('nyc'), ['nyc.trades'])

fs.rmSync(warehouse, { recursive: true, force: true })

### The object model: namespaces of tables

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { iceberg } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const catalog = new iceberg.Catalog(path.join(root, 'warehouse'))

// The views are lazy: an empty warehouse answers empty, touching nothing.
assert.equal(catalog.namespaces.size(), 0)
const sales = catalog.namespaces.openOrCreate('sales')

// The write conveniences create a table on first write, from the rows'
// own schema; the views chain a catalog to a namespace to a table.
sales.tables.append(
  'orders',
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    qty: arrow.vectorFromArray([5, 2.5], new arrow.Float64()),
  }),
)
assert.ok(sales.tables.has('orders'))
assert.deepEqual(sales.tables.names(), ['orders'])

const table = catalog.namespaces.get('sales').tables.get('orders')
assert.equal(table.scan().toTable().numRows, 2)

// A nested namespace is reached through its parent's own view.
sales.namespaces.create('eu')
assert.deepEqual(catalog.namespaces.get('sales').namespaces.names(), ['eu'])

// A missing name is refused naming it, never answered as an empty table.
assert.throws(() => catalog.namespaces.get('marketing'), /marketing/)

fs.rmSync(root, { recursive: true, force: true })

## Data files aim at a size

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, iceberg } = require('yggdryl')

const warehouse = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const catalog = new iceberg.Catalog(warehouse)

// The default target is Iceberg's own 512 MiB.
const table = catalog.createTable('tiny.rows', [Field.from('id: int64')])
assert.equal(table.targetFileSize, 512 * 1024 * 1024)

// Five appends, five snapshots, five small files.
for (const value of [0n, 1n, 2n, 3n, 4n]) {
  table.append(new arrow.Table({ id: arrow.vectorFromArray([value], new arrow.Int64()) }))
}
assert.equal(table.inspectFiles().toTable().numRows, 5)

// Compaction rewrites the small groups as one replace commit and reports it.
const compaction = table.compact()
assert.equal(compaction.filesBefore, 5)
assert.equal(compaction.filesAfter, 1)
assert.ok(compaction.bytesRewritten > 0)
assert.equal(table.scan().toTable().numRows, 5)

// Nothing to do is a no-op that commits nothing.
assert.deepEqual(table.compact(), { filesBefore: 0, filesAfter: 0, bytesRewritten: 0 })

fs.rmSync(warehouse, { recursive: true, force: true })

## One options value, three layers

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')
const table = iceberg.Table.create(root, schema)

// Nothing set: every field answers its documented default.
assert.equal(table.options().commitRetries, 4)
assert.equal(table.options().targetFileSize, 512 * 1024 * 1024)

// The property layer is the table's own metadata, one commit away.
table.updateProperties({ 'commit.retry.num-retries': '9' })
assert.equal(table.options().commitRetries, 9)

// An explicit override shadows the property on this handle alone; nothing
// is written, and an unset field still resolves the other layers.
table.setOptions(new iceberg.IcebergOptions({ commitRetries: 2 }))
assert.equal(table.options().commitRetries, 2)
assert.equal(table.options().commitMinBackoffMs, 100)

// The trailing argument is the per-call layer: this write alone is sized.
const rows = new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) })
table.append(rows, new iceberg.IcebergOptions({ targetFileSize: 1 << 20 }))
assert.equal(table.options().targetFileSize, 512 * 1024 * 1024)

// A value the core refuses is refused at the boundary, naming it.
assert.throws(() => new iceberg.IcebergOptions({ targetFileSize: 0 }))

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## The data file format

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')
const table = iceberg.Table.create(root, schema)

const rows = (id) =>
  new arrow.Table({ id: arrow.vectorFromArray([id], new arrow.Int64()) })

// One Parquet append, then one Avro append - the option is the trailing
// argument every write already takes.
table.append(rows(1n))
table.append(rows(2n), new iceberg.IcebergOptions({ dataFormat: 'avro' }))

const formats = table.dataFiles().map((file) => file.fileFormat).sort()
assert.deepEqual(formats, ['AVRO', 'PARQUET'])
assert.equal(table.scan().toTable().numRows, 2)

// Stored per table, the spec's own key configures every writer.
table.updateProperties({ 'write.format.default': 'avro' })
assert.equal(table.options().dataFormat, 'AVRO')

// A format the build cannot encode is named before anything is written.
assert.throws(
  () => table.append(rows(3n), new iceberg.IcebergOptions({ dataFormat: 'orc' })),
  /ORC/,
)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Branches and tags

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')
const table = iceberg.Table.create(root, schema)

const rows = (id) =>
  new arrow.Table({ id: arrow.vectorFromArray([id], new arrow.Int64()) })

table.append(rows(1n))
const audited = table.currentSnapshot.snapshotId

// The tag pins the audited state; the table keeps moving.
table.createTag('audit-2026', audited)
table.append(rows(2n))
table.createBranch('review', audited)

// Every ref reads as the complete table it names.
assert.equal(table.scanRef('audit-2026').toTable().numRows, 1)
assert.equal(table.scanRef('review').toTable().numRows, 1)
assert.equal(table.scan().toTable().numRows, 2)

// A branch fast-forwards only along its own ancestry: the target must reach
// the branch's head by parent ids, so no history can be lost.
const head = table.currentSnapshot.snapshotId
table.fastForward('review', head)
assert.equal(table.snapshotByRef('review').snapshotId, head)

// Removing a ref reports what it pointed at; the snapshots stay retained,
// and a second removal is refused rather than committing nothing.
assert.equal(table.removeRef('review').snapshotId, head)
assert.throws(() => table.removeRef('review'), /review/)

// Expiry honors every ref's retention: the tagged snapshot survives a
// cutoff that would otherwise expire everything old.
assert.deepEqual(table.expireSnapshots(Number.MAX_SAFE_INTEGER), [])
assert.equal(table.snapshots.length, 2)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Schema evolution and field ids

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, schema)
table.append(new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) }))

// Add a column. Numbering continues above `last-column-id`, so the new column
// can never be confused with a dropped one.
const evolved = fields.struct('row', [Field.from('id: int64'), Field.from('quantity: int64')], {
  nullable: false,
})
assert.equal(table.evolveSchema(evolved), 1, "the new schema's id")

// The old schema is retained, so the snapshot written under it still reads.
assert.equal(table.schemas.length, 2)
assert.equal(table.schemas[0].dataType.length, 1)

// And the file written before the column existed reads it as null.
const rows = table.scan().toTable()
assert.deepEqual(rows.schema.fields.map((child) => child.name), ['id', 'quantity'])
assert.equal(rows.getChild('quantity').nullCount, rows.numRows)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

In [ ]:
const assert = require('node:assert/strict')
const { Field, fields, iceberg } = require('yggdryl')

const leg = fields.struct('leg', [Field.from('price: decimal(18, 4)')])
const plain = fields.struct('row', [Field.from('id: int64'), leg], { nullable: false })

// Depth first from `start`; the numbered schema is what comes back, so the
// schema handed in is left as it was.
const schema = iceberg.assignFieldIds(plain)
assert.equal(plain.dataType.at(0).parquetFieldId, null)
assert.equal(schema.dataType.at(0).parquetFieldId, 1)
assert.equal(schema.dataType.at(1).parquetFieldId, 2)
assert.equal(schema.dataType.at(1).dataType.at(0).parquetFieldId, 3)

// The root is not a column, so it is not numbered.
assert.equal(schema.parquetFieldId, null)

// A field that already carries an id keeps it, so a second pass changes nothing.
assert.equal(iceberg.assignFieldIds(schema, 100).dataType.at(0).parquetFieldId, 1)

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { Field, fields, iceberg } = require('yggdryl')

// A plain schema carries no ids; creating the table numbers it.
const unnumbered = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')

const table = iceberg.Table.create(root, unnumbered)
assert.equal(table.schema.dataType.at(0).parquetFieldId, 1)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Evolving a schema

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { Field, fields, iceberg } = require('yggdryl')

// Legal promotions pass; anything else is refused naming both sides.
iceberg.canPromote('int32', 'int64')
iceberg.canPromote('decimal128(10, 2)', 'decimal128(18, 2)')
assert.throws(() => iceberg.canPromote('int64', 'int32'), /int64 to int32/)

const declared = fields.struct('row', [Field.from('id: int32'), Field.from('symbol: utf8')], {
  nullable: false,
})
const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-')), 'trades')
const table = iceberg.Table.create(root, declared)

// Widen id, rename symbol, add venue - one evolved schema, one commit.
const schemaId = table
  .updateSchema()
  .updateType('id', 'int64')
  .renameColumn('symbol', 'ticker')
  .addColumn('', 'venue: utf8')
  .commit()
assert.equal(schemaId, 1)

const evolved = table.schema
assert.deepEqual(Array.from(evolved.dataType, (child) => child.name), ['id', 'ticker', 'venue'])
assert.equal(String(evolved.dataType.at(0).dataType), 'int64')
// A renamed column keeps its identifier: the name is a label, the id is the column.
assert.deepEqual(Array.from(evolved.dataType, (child) => child.parquetFieldId), [1, 2, 3])

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## Schemas as documents

In [ ]:
const assert = require('node:assert/strict')
const { iceberg, json } = require('yggdryl')

const document = json.loads(
  Buffer.from(`{"type":"struct","schema-id":0,"fields":[
    {"id":1,"name":"id","required":true,"type":"long"},
    {"id":2,"name":"symbol","required":false,"type":"string"}
  ]}`),
)

// An Iceberg schema is a non-null struct field; its columns are the children.
const schema = iceberg.schemaFromJson('row', document)
assert.equal(schema.dataType.kind, 'struct')
assert.equal(schema.nullable, false)
assert.equal(schema.dataType.length, 2)
assert.equal(String(schema.dataType.at(0).dataType), 'int64')

// `required` inverts into nullability, and `id` becomes PARQUET:field_id.
assert.equal(schema.dataType.at(0).nullable, false)
assert.equal(schema.dataType.at(1).nullable, true)
assert.equal(schema.dataType.at(0).parquetFieldId, 1)
assert.equal(schema.dataType.at(0).get('PARQUET:field_id'), '1')

// The same document comes back out.
assert.deepEqual(iceberg.schemaToJson(schema).asJs(), document)